##  HW2 Part 2 - Flatting JSON data from Yelp

For this second part of the assignment we're going to be using the Yelp API via the python package `yelpapi`.  Like with the Spotify API, you need to go get a yelp developer account here: https://www.yelp.com/login?return_url=/developers/v3/manage_app.
When you try to create an app, it asks for name, description and contact info. The it gives you clientid and api key.

The goal will be to make a dataset of local taco places that are doing well during Coronavirus. Specifically, we want to make a dataset that takes their average score since they've been open and compares it to the average score of the last three reviews.  Shops that have been doing well should hopefully have these two averages be similar.

There will be three main steps to this process:

* First you'll search by location type and get aggregate information for everything that falls in the ice cream category.
* Second you'll get reviews for all those locations.  Yelp only returns three reviews when you call an ID, but that still works. Given you can only query one ID at a time, you'll need to write a loop to create a dataframe of all the reviews.  
* Third, you'll aggregate the latest review information to see how their average review score compares to their overall average review score.  

In [1]:
# Mount google drive
import os, sys
from google.colab import drive
drive.mount('/content/mnt')
nb_path = '/content/notebooks'
os.symlink('/content/mnt/My Drive/Colab Notebooks', nb_path)
sys.path.insert(0, nb_path)  # or append(nb_path)

Mounted at /content/mnt


In [2]:
# Install yelpapi once. Next time you run this notebook, you can skip this.
!pip install --target=$nb_path yelpapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 14.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.3, but you have requests 2.32.4 which is incompatible.


In [17]:
# Now enter your API key so you can make requests
from yelpapi import YelpAPI

# yelp_api = YelpAPI('ENTER YOUR KEY HERE')
api_key = 'UKDKvqNbOYV4yG2VeYBzKRjbsfFCOUjyZx1HXwcw_cEWMsBpXHWcxm0i9uFJWgKI8Y6Hxkzo2lc7Rg_gMWUl_XQiTYZjWBwmqvI-4vU--3LJH-zyLfHLoNZ0ddOGaHYx'
yelp_api = YelpAPI(api_key)


### Making calls to with Yelp API

There are many functions in the `yelpapi` package.  The first one we'll use is `search_query()`.  You can put in a term you want to search for followed up by the location and it'll give you all the locations that match.  For example, the following would search for all ice cream places here in Tucson.

```
ice = yelp_api.search_query(term = 'ice cream', location = 'Tucson, AZ')
```

This will give you a response of all the ice cream places in Tucson. It'll be in JSON form so will need flattening before being useful.

### Q6 Getting all taco shops and flattening - [3 points]

**Task:** Start by making a dataframe that uses the `search_query()` function to search using the term 'taco'.  Call this `taco_shops`.  After that, flatten the json results to `taco_shops_df`.

In [18]:
## Q6 part 1 Your code starts here
# search for taco shops and store to taco_shops
taco_shops = yelp_api.search_query(term='taco',  location='Tucson, AZ')

In [19]:
# Look at taco_shops
display(taco_shops)

{'businesses': [{'id': 'ZPg-3g_qfPqWoh1vWvX65A',
   'alias': 'casa-asada-taqueria-y-cerveceria-tucson-5',
   'name': 'Casa Asada Taqueria y Cerveceria',
   'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/67XlCtGCdmgHHycAbO8N7w/o.jpg',
   'is_closed': False,
   'url': 'https://www.yelp.com/biz/casa-asada-taqueria-y-cerveceria-tucson-5?adjust_creative=_vvMtBeRO_A5qdAVAryhyg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=_vvMtBeRO_A5qdAVAryhyg',
   'review_count': 82,
   'categories': [{'alias': 'tacos', 'title': 'Tacos'},
    {'alias': 'desserts', 'title': 'Desserts'},
    {'alias': 'cocktailbars', 'title': 'Cocktail Bars'}],
   'rating': 4.0,
   'coordinates': {'latitude': 32.252208443091654, 'longitude': -110.9436083},
   'transactions': ['delivery', 'pickup'],
   'location': {'address1': '2502 N Campbell Ave',
    'address2': None,
    'address3': '',
    'city': 'Tucson',
    'zip_code': '85719',
    'country': 'US',
    'state': 'AZ',
    'display_addres

In [20]:
# What keys are present?
taco_shops.keys()
## Q6 part 1  Your code ends here - Any code outside of these start/end markers won't be graded

dict_keys(['businesses', 'total', 'region'])

Now use the json_normalize function to flatten `taco_shops`.  Note that you need to select the key that contains the businesses when flattening.  But this one is easier than the examples from the homework in that you don't need to provide any other arguments.  

Store the result as `taco_shops_df`

After that select only the columns 'id', 'alias', 'name', 'review_count', and 'rating'.

In [21]:
from pandas import json_normalize
## Q6 part 2 Your code starts here
# Flatten to taco_shops_df
taco_shops_df = json_normalize(taco_shops, record_path='businesses')
taco_shops_df.head() # Check

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,...,location.city,location.zip_code,location.country,location.state,location.display_address,attributes.business_temp_closed,attributes.menu_url,attributes.open24_hours,attributes.waitlist_reservation,price
0,ZPg-3g_qfPqWoh1vWvX65A,casa-asada-taqueria-y-cerveceria-tucson-5,Casa Asada Taqueria y Cerveceria,https://s3-media0.fl.yelpcdn.com/bphoto/67XlCt...,False,https://www.yelp.com/biz/casa-asada-taqueria-y...,82,"[{'alias': 'tacos', 'title': 'Tacos'}, {'alias...",4.0,"[delivery, pickup]",...,Tucson,85719,US,AZ,"[2502 N Campbell Ave, Tucson, AZ 85719]",None,https://asadataqueria.com/wp-content/uploads/2...,NaN,None,NaN
1,jmwasbZfgj3honf79qKsnA,street-taco-and-beer-co-tucson,STREET - Taco & Beer Co.,https://s3-media0.fl.yelpcdn.com/bphoto/VOaRTn...,False,https://www.yelp.com/biz/street-taco-and-beer-...,1055,"[{'alias': 'mexican', 'title': 'Mexican'}, {'a...",4.3,[delivery],...,Tucson,85701,US,AZ,"[58 W Congress St, Tucson, AZ 85701]",None,http://street.restaurant/#s-8aa5f6be-1fb5-4101...,NaN,None,$$
2,nvwtUKh6rBb29rFbC9kq5w,taco-rico-tucson-2,Taco Rico,https://s3-media0.fl.yelpcdn.com/bphoto/f9mKwX...,False,https://www.yelp.com/biz/taco-rico-tucson-2?ad...,121,"[{'alias': 'mexican', 'title': 'Mexican'}, {'a...",4.9,[],...,Tucson,85741,US,AZ,"[4500 W Ina Rd, Tucson, AZ 85741]",None,https://tacoricoaz.com/menu,NaN,None,$
3,LQcGL4hfJAeK6bk2ZdhmXw,aqui-con-el-nene-tucson,Aqui Con El Nene,https://s3-media0.fl.yelpcdn.com/bphoto/BczDqo...,False,https://www.yelp.com/biz/aqui-con-el-nene-tucs...,473,"[{'alias': 'mexican', 'title': 'Mexican'}, {'a...",4.4,"[delivery, pickup]",...,Tucson,85705,US,AZ,"[4415 N Flowing Wells Rd, Tucson, AZ 85705]",None,None,NaN,None,$$
4,uJEhFxXTf58g8MHjX8ZVqg,del-rancho-carniceria-and-tacos-tucson,Del Rancho Carniceria & Tacos,https://s3-media0.fl.yelpcdn.com/bphoto/fLX7Oe...,False,https://www.yelp.com/biz/del-rancho-carniceria...,12,"[{'alias': 'tacos', 'title': 'Tacos'}, {'alias...",4.0,[],...,Tucson,85741,US,AZ,"[4505 W Ina Rd, Tucson, AZ 85741]",None,None,NaN,None,NaN


In [22]:
# Select only necessary columns
taco_shops_df = taco_shops_df[['id', 'alias', 'name', 'review_count', 'rating']]
taco_shops_df.shape # Check shape.  Is it 20 x 5?
## Q6 part 2  Your code ends here - Any code outside of these start/end markers won't be graded

(20, 5)

## Q7 Getting reviews for the taco shops - [6 points]

Now we're going to use the `reviews_query()` function to get the last three reviews for a given ID.  The issue here is that you can only feed it one ID at a time.  So, we'll have to write a loop that queries for each ID in `taco_shops_df` and builds out a dataframe of reviews.  

**Task:**  Write a for loop that does the following steps:
* First make an empty data frame outside of the loop called `taco_shops_reviews_df`
* Initalize your loop so that it runs the length of `taco_shops_df` you made earlier.
* For each i in loop, use the id from `taco_shops_df` to get reviews from yelp using the `reviews_query()` function in `yelp_api` and store it to an object called `reviews`.
* Flatten `reviews` to an objected called `reviews_df`
* Add `location_id` to `reviews_df` - I gave you the code to do this :)
* Append `reviews_df` to `taco_shops_reviews_df` such that it builds out that dataframe with each `reviews_df` dataframe that's generated each loop.
* After the loop is done select only the columns `'id', 'text', 'rating', 'time_created', 'location_id'`

In [24]:
## Q7 Your code starts here
# Write our loop
import pandas as pd
taco_shops_reviews_df = pd.DataFrame()
for i in range(len(taco_shops_df)):
  reviews = yelp_api.reviews_query(id=taco_shops_df['id'][i])
  reviews_df = json_normalize(reviews, record_path='reviews')
  reviews_df['location_id'] = taco_shops_df['id'][i]
  taco_shops_reviews_df = pd.concat([taco_shops_reviews_df, reviews_df])


HTTPError: 404 Client Error: Not Found for url: https://api.yelp.com/v3/businesses/ZPg-3g_qfPqWoh1vWvX65A/reviews

In [15]:
# Select columns 'id', 'text', 'rating', 'time_created', 'location_id'
taco_shops_reviews_df = taco_shops_reviews_df[['id', 'text', 'rating', 'time_created', 'location_id']]
## Q7  Your code ends here - Any code outside of these start/end markers won't be graded

KeyError: "None of [Index(['id', 'text', 'rating', 'time_created', 'location_id'], dtype='object')] are in the [columns]"

### Q8 Aggregating your review data - [3 points]

**Task:** Now go and do a data aggregation to get the mean review score across the three reviews. Remember, we want this grouped by location_id.  Call this dataframe `latest_reviews_agg`.

In your groupby you should set `as_index` to false to make joining on the ID values easier.

You should also rename the second column to `mean_rating` vs. leaving it as the dual level name.


In [ ]:
## Q8 Your code starts here
# Do your groupby to get mean rating.
# Call it latest_reviews_agg.
... = ...(..., as_index=False).....

In [ ]:
# Check it!
...

In [ ]:
# Rename so the two column names are 'location_id' and 'mean_rating'
... = ...
## Q8  Your code ends here - Any code outside of these start/end markers won't be graded

### Q9 Join your two datasets and one last transform - [3 points]

**Task:** Now it's time to join `latest_reviews_agg` back to your `taco_shops_df` dataframe. You're going to want to join them on the location_id, but remember it is called just 'id' in the `taco_shops_df`  dataframe. You can solve this either by renaming the columns before merging or using the 'left_on' and 'right_on' parameters of .merge. However you choose to do it, the merged table should only include 1 column containing this data and it should be called 'location_id'. Call this joined dataset `taco_shops_comp`

After you make that dataset do one last transform. In this I want you to make a new column called 'still_good' where the value is 'yes' if the mean_rating is greater than or equal to the average rating since they opened, or 'no' if the rating has dropped. The idea here is that this could be a value that one would use to see if their average score of the latest reviews has improved or suffered during Covid.  

In [ ]:
## Q9 Your code starts here
# Join and name taco_shops_comp

...

When you are comparing the columns make sure to not comparing taco_shops_df and taco_shops_comp since their tuples may have different orders! Use both columns from taco_shops_comp

In [ ]:
# Make still_good column
...

In [ ]:
# So, do any locations have a lower average in their last three reviews compared to their average overall score?
...
## Q9  Your code ends here - Any code outside of these start/end markers won't be graded

You should get 9 shops (this may change suddenly since data is live, but on July 12 2024 is still 9) including El Rustico and Tumerico